# 08's PPO on the MLX stack

`08` moved generation to MLX: mlx-lm samples the rollouts and evaluations, with the policy's LoRA folded into its weights before each rollout. The rest of a step stayed in PyTorch on MPS -- the update, the reference's log-probabilities, the separate critic's reads -- about 85% of it. This notebook is 08, copied, with all of it on MLX, and no PyTorch in the process (`--trainer mlx`):

| | 08 | 08.5 |
|---|---|---|
| rollouts and evaluations | MLX | MLX |
| the update (LoRA, PPO's actor loss, AdamW) | PyTorch, MPS | MLX |
| the reference's log-probabilities (KL to the seed) | PyTorch, MPS | MLX |
| the separate critic's reads (Qwen3-4B, layer 27) | PyTorch, MPS | MLX |
| masks, GAE by the turn, the prover's progress, the critic's closed-form heads | PyTorch | MLX (the heads float64, on MLX's CPU stream) |

What it took (`08.5_mlx_stack.py`, the section "08.5: the MLX stack"):

* **Qwen3.5's gated delta rule, differentiable.** mlx-lm's Metal kernel has no backward, and its training path is a per-token loop that keeps every token's recurrent state for the backward -- 16 × 128 × 128 float32 per token per layer, ~90 GB for one 5.4k-token episode. `mlx_chunk_gated_delta_rule` is 08's chunked MPS rule (64 tokens a chunk, float32 inside) in MLX ops, left to MLX's autograd.
* **Gradient checkpointing per decoder layer** (`mlx_checkpoint_layers`): without it the full-attention layers' T × T scores of every layer are kept for the backward (a 7.9k-token episode peaked at 55 GB); with it, under 10 GB.
* **LoRA in PEFT's format both ways** (`MLXTrainable`): mlx-lm's `LoRALinear` stores A and B transposed and defaults to scale 20; here lora_a = Aᵀ, lora_b = Bᵀ, scale lora_alpha / r = 2, and checkpoints are written as PEFT's, so `probe` and everything else read them as before.
* **AdamW as PyTorch's**: MLX's Adam has no bias correction unless asked.
* **The critic on mlx-lm** (`MLXSeparateCritic`): the residual stream after layer 27 at the last token before each turn, the system turn read once and its cache copied per episode, as 08's.
* **The bookkeeping without PyTorch** (`mx_batch`, `mx_gae_turn`, `mx_normalized_advantages`, `MXCriticHeads`, ...): 08's functions, the same computation over MLX arrays. `train_ppo_mlx` runs under `block_torch`. Transformers then reports "PyTorch was not found" and loads its tokenizers only, any `import torch` would raise, and the run ends by printing whether PyTorch was loaded (it was not).

One trap on the way, silent: in MLX 0.32.3 the backward of the strided slices `inv[..., 0::2]` / `inv[..., 1::2]` in the triangular inverse came out wrong -- forward right, gradient cosine 0.56 against float64. Taking each pair by a reshape fixed it.

**The run.** D37: 08's configuration with three of the 37 PPO implementation details (advantages normalized per minibatch, 4 minibatches × 4 epochs, gradient clip 0.5, Adam eps 1e-5, lr 2e-6 annealed linearly to 0), 200 steps. It starts from 07's seed and 07's first 25 steps (the same questions at every step), so it pairs step by step with 07's run and 08's first run (v1). It was first started in PyTorch and stopped at step 29, then started again here, on MLX from the start.


In [1]:
import importlib.util, json, sys
from pathlib import Path

HERE = Path.cwd() if (Path.cwd() / "08.5_mlx_stack.py").exists() else Path("experiments/notebooks/smoke_test35")
# This notebook's module (08's, copied and changed as its header says), loaded by path. Nothing below needs a GPU:
# every number is read from the records.
spec = importlib.util.spec_from_file_location("pc085", HERE / "08.5_mlx_stack.py")
pc = importlib.util.module_from_spec(spec)
sys.modules["pc085"] = pc
spec.loader.exec_module(pc)
RUNS, RUNS08, RUNS07 = pc.RUNS, HERE / "08_prover_critic_runs", pc.RUNS07
PPO07, PPO08_V1, PPO08_D37 = "ppo-n200-r100-turn-s0", "ppo-qwen3-4b-bo4-a0.8-s0", "ppo-qwen3-4b-bo4-a0.8-d37-s0"
PPO085 = "ppo-mlx-qwen3-4b-bo4-a0.8-d37-s0"


def rows_of(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines() if line.strip()]


def jload(path):
    path = Path(path)
    return json.loads(path.read_text()) if path.exists() else None


def median(xs):
    xs = sorted(xs)
    return (xs[(len(xs) - 1) // 2] + xs[len(xs) // 2]) / 2 if xs else float("nan")

## Checks against 08's PyTorch pieces

`08.5_mlx_stack.py trainer-check`, on MPS and MLX in one process (`08.5_mlx_stack_trainer_checks.json`):

1. **The rule**, on one episode's sizes (T 5.4k, 16 heads of 128), in float32: the output and the gradient of every input, MLX's chunked rule against 08's MPS one, and the time of a forward + backward.
2. **The inverse's gradient** against `torch.linalg.inv` in float64 (the trap above).
3. **The policy**: 07's seed (the run's start), on recorded episodes of 08's first run spread over a step's lengths.
   - Each policy token's log-probability, PyTorch's against MLX's.
   - The gradient of their sum with respect to every LoRA tensor.
   - One episode's forward + backward time: PyTorch with gradient checkpointing, as `train_ppo` loads it, against MLX with its layers checkpointed.

   bf16 both ways, so the two differ at bf16's rounding.
4. **The critic's features**, per turn.

In [2]:
CHK = jload(HERE / "08.5_mlx_stack_trainer_checks.json")
if CHK is None:
    print("trainer-check not run yet")
else:
    r = CHK["delta_rule_float32"]
    print(f"mlx {CHK['mlx']}, torch {CHK['torch']}; the rule's chunk {CHK['chunk']}")
    print(f"1. the rule, T={r['tokens']}, float32: output cosine {r['output']['cosine']:.7f} (max |diff| {r['output']['max_abs_diff']:.1e})")
    for k in ("q", "k", "v", "g", "beta"):
        x = r[f"grad_{k}"]
        print(f"   gradient of {k:4}: cosine {x['cosine']:.7f}, max |diff| {x['max_abs_diff']:.1e} (largest {x['scale']:.2f})")
    s = r["seconds_forward_backward"]
    print(f"   forward + backward: PyTorch MPS {min(s['torch_mps']):.3f}s, MLX {min(s['mlx']):.3f}s (best of 3)")
    x = CHK["inverse_grad_vs_float64"]
    print(f"2. the inverse's gradient against float64: cosine {x['cosine']:.8f}, max |diff| {x['max_abs_diff']:.1e}")
    print("3. the policy (07's seed), recorded episodes:")
    for p in CHK["policy"]:
        s = p["seconds_forward_backward"]
        print(f"   {p['tokens']:5} tokens ({p['policy_tokens']:4} the policy's): log-prob sums {p['logprob_sum_torch']:8.2f} / "
              f"{p['logprob_sum_mlx']:8.2f}, per token mean |diff| {p['logprob_mean_abs_diff']:.4f} (max {p['logprob_max_abs_diff']:.3f}) | "
              f"gradient cosine median {p['grad_cosine_median']:.4f} (min {p['grad_cosine_min']:.4f}), norm ratio {p['grad_norm_ratio_median']:.3f} | "
              f"forward + backward {s['torch_mps']:.1f}s / {s['mlx']:.1f}s")
    print("4. the critic's features (Qwen3-4B, layer 27):")
    for c in CHK["critic"]:
        print(f"   {c['turns']} turns: cosine per turn mean {c['cosine_mean']:.5f} (min {c['cosine_min']:.5f}), norm ratio {c['norm_ratio']:.4f}")

mlx 0.32.3, torch 2.14.1; the rule's chunk 64
1. the rule, T=5400, float32: output cosine 1.0000207 (max |diff| 2.9e-07)
   gradient of q   : cosine 1.0000199, max |diff| 1.6e-06 (largest 1.04)
   gradient of k   : cosine 1.0000249, max |diff| 1.7e-06 (largest 1.11)
   gradient of v   : cosine 1.0000250, max |diff| 2.6e-07 (largest 0.10)
   gradient of g   : cosine 1.0000000, max |diff| 8.6e-07 (largest 0.67)
   gradient of beta: cosine 0.9999999, max |diff| 1.1e-06 (largest 0.86)
   forward + backward: PyTorch MPS 0.072s, MLX 0.283s (best of 3)
2. the inverse's gradient against float64: cosine 1.00000000, max |diff| 1.5e-05
3. the policy (07's seed), recorded episodes:
    3891 tokens ( 756 the policy's): log-prob sums   -55.50 /   -55.14, per token mean |diff| 0.0022 (max 0.310) | gradient cosine median 0.9858 (min 0.9464), norm ratio 0.962 | forward + backward 4.9s / 5.3s
    4341 tokens (1136 the policy's): log-prob sums   -38.15 /   -38.22, per token mean |diff| 0.0012 (max 0.125)

In [3]:
# 5. train_ppo_mlx's bookkeeping against 08's PyTorch functions (08.5_mlx_stack.py bookkeeping-check): two batches of
# 8 recorded episodes with random values and rewards, the critic's heads on a synthetic window of PPO's size
BK = jload(HERE / "08.5_mlx_stack_bookkeeping_checks.json")
if BK is None:
    print("bookkeeping-check not run yet")
else:
    print(f"5. the bookkeeping, MLX against PyTorch ({BK['episodes']} recorded episodes): largest absolute difference")
    for k, v in BK["max_abs_diff"].items():
        print(f"   {k:34} {v:.1e}")

5. the bookkeeping, MLX against PyTorch (16 recorded episodes): largest absolute difference
   batch_mask                         0.0e+00
   batch_answer                       0.0e+00
   batch_turn_end                     0.0e+00
   batch_content                      0.0e+00
   batch_rewards                      0.0e+00
   batch_turn                         0.0e+00
   gae_turn_advantages                4.8e-07
   gae_turn_returns                   6.0e-07
   gae_advantages                     9.5e-07
   normalized_advantages_batch        2.4e-07
   normalized_advantages_minibatch    1.5e-08
   per_turn_tokens                    0.0e+00
   prover_progress                    0.0e+00
   separate_critic_stats              1.3e-08
   critic_fit                         0.0e+00
   critic_heads_value                 8.1e-07
   critic_heads_success               3.4e-08


## The run: D37 on MLX, against 07 and 08's v1

The same seed and 07's first 25 steps: every run sees the same questions at every step, so the sampled rewards pair by step. The evaluations' 60 dev questions are the same for all; 07's are PyTorch-generated, v1's from step 80 on and 08.5's MLX-generated.


In [4]:
def load_run(d):
    if not (d / "metrics.jsonl").exists():
        return None
    return {"m": rows_of(d / "metrics.jsonl"), "ev": rows_of(d / "eval.jsonl") if (d / "eval.jsonl").exists() else [],
            "end": jload(d / "ending.json")}


for label, d in (("07, PyTorch", RUNS07 / PPO07), ("08 v1, PyTorch (MLX generation from step 80)", RUNS08 / PPO08_V1), ("08.5 D37, MLX", RUNS / PPO085)):
    run = load_run(d)
    if run is None:
        print(f"{label}: not run yet")
        continue
    m, end = run["m"], run["end"]
    act = [r for r in m if not r["critic_warmup"]]
    own = [r for r in m if not r.get("warm_from")]  # the steps this run took itself
    print(f"{label}: {len(m)} steps" + (f"; ended at {end['stopped_at']} ({end['reason']}), best evaluation {end['best_reward']:.4f} at step {end['best_step']}"
                                        if end else " (running)"))
    if act:
        def mean(k, spec):
            xs = [r[k] for r in act if r.get(k) is not None]
            return format(sum(xs) / len(xs), spec) if xs else "-"
        print(f"   actor steps {len(act)}: reward {mean('reward_mean', '.3f')} decision {mean('decision_correct', '.3f')} evidence {mean('evidence', '.3f')} "
              f"| recovered {mean('rec_recovered_given_error', '.3f')} | kl {mean('kl_mean', '.4f')} | approx_kl max {mean('approx_kl_max', '.4f')} "
              f"| first ratio {mean('ratio_first', '.4f')} | clip frac {mean('clip_frac', '.3f')} | optimizer steps {mean('updates', '.0f')}")
    if own:
        print(f"   time per step (median of {len(own)}): {median([r['step_seconds'] for r in own]):.0f}s, generation "
              f"{median([r['gen_seconds'] for r in own]):.0f}s; actor steps only {median([r['step_seconds'] for r in act if not r.get('warm_from')] or [float('nan')]):.0f}s")
    for r in run["ev"]:
        g = r.get("recovery", {}).get("recovered_given_error")
        print(f"   eval @{r['step']:>4}: reward {r['reward']:.4f} decision {r['decision_correct']:.3f} evidence {r['evidence']:.3f} "
              f"| recovered {'-' if g is None else f'{g:.3f}'}" + ("  (07's)" if r.get("reused") else ""))

07, PyTorch: 120 steps; ended at 120 (step budget 120 reached), best evaluation 0.6730 at step 120
   actor steps 95: reward 0.538 decision 0.516 evidence 0.467 | recovered 0.452 | kl 0.0019 | approx_kl max - | first ratio - | clip frac 0.001 | optimizer steps -
   time per step (median of 120): 446s, generation 146s; actor steps only 474s
   eval @   0: reward 0.5856 decision 0.550 evidence 0.500 | recovered 0.400  (07's)
   eval @  40: reward 0.6580 decision 0.633 evidence 0.600 | recovered 0.448
   eval @  80: reward 0.6479 decision 0.633 evidence 0.583 | recovered 0.438
   eval @ 120: reward 0.6730 decision 0.683 evidence 0.633 | recovered 0.500
08 v1, PyTorch (MLX generation from step 80): 120 steps; ended at 120 (step budget 120 reached), best evaluation 0.6792 at step 40
   actor steps 95: reward 0.528 decision 0.499 evidence 0.443 | recovered 0.398 | kl 0.0017 | approx_kl max 0.0002 | first ratio 1.0000 | clip frac 0.001 | optimizer steps 4
   time per step (median of 95): 483s

## Summary

*(Written when the runs are in. Set before them:)*

1. **Same numbers?** Against 08's PyTorch pieces (the checks): the rule to float32's rounding; the policy's log-probabilities and gradients, and the critic's features, to bf16's.
2. **Learning.** 08.5's D37 run against 07 and 08's v1, paired by step: the greedy evaluations at steps 40-200, the sampled reward, the update's measures (approx_kl, first ratio, clip fraction), the KL to the seed.
3. **Faster?** A step's time on each stack, and where it goes.